# Scenario engine construction for the Mujib Basin Digital Twin

This notebook builds the regression-based scenario emulator described in Section 3.4.2 of the thesis.
It takes the prepared SWAT annual dataset (from notebook 01) and produces the scenario JSON file
that the dashboard loads at runtime.

**Processing steps:**

1. Load the prepared SWAT annual dataset (71 sub-basins, 1988-2020)
2. Compute the monthly climatology for each sub-basin
3. Fit ridge regressions per sub-basin: PRECIP, PET to SURQ; PRECIP, PET to PERC; log(SURQ) to log(SYLD)
4. Apply climate perturbations (3 x 3 grid: precipitation 0%, +10%, +20%; temperature 0, +1, +2 C)
5. Apply intervention multipliers (Baseline, Marab, Vallerani, Combined) from Table 3.4
6. Add CMIP6 near-term proxy cases (SSP2-4.5 and SSP5-8.5) using annual regression and monthly scaling
7. Export the assembled scenario JSON for the dashboard

**Inputs:** `swat-prepared/output_sub_FULL_FIXED2020.parquet`, existing base scenario JSON

**Outputs:** `runtime-data/scenarios/scenarios_USED_BY_CESIUM_FINAL_71_CMIP6_PROXY.json`

**Thesis reference:** Section 3.4.2 (methodology), Section 4.6 (results), Table 3.4 (multipliers)


## 1. Configuration and imports


In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from copy import deepcopy

# Repository paths - adjust these to match your local setup
REPO_ROOT = Path("..")
SWAT_PARQUET = REPO_ROOT / "swat-prepared" / "output_sub_FULL_FIXED2020.parquet"
SCENARIO_JSON_OUT = REPO_ROOT / "runtime-data" / "scenarios" / "scenarios_USED_BY_CESIUM_FINAL_71_CMIP6_PROXY.json"

# Columns needed from the SWAT parquet
COLS_NEEDED = ["SUB", "MON", "AREAkm2", "PRECIPmm", "PETmm", "SURQmm", "SYLDt_ha", "PERCmm", "ETmm", "WYLDmm"]

# Output directory for intermediate files
OUT_DIR = REPO_ROOT / "swat-prepared"
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load the prepared SWAT dataset

The annual dataset was prepared in notebook 01 from ICARDA's 2024 ArcSWAT model outputs.
It contains 2,343 records (71 sub-basins x 33 years, 1988-2020).


In [ ]:
df = pd.read_parquet(SWAT_PARQUET)

missing = [c for c in COLS_NEEDED if c not in df.columns]
if missing:
    print("Missing required columns:", missing)
else:
    print(f"Loaded {len(df)} records, {df['SUB'].nunique()} sub-basins")
    print(f"Year range: {df['MON'].min()} to {df['MON'].max()}")
    print(f"Columns: {list(df.columns)}")


## 3. Compute monthly climatology

For each sub-basin, compute the mean value of each indicator across all years
for each calendar month. This produces a 71 x 12 climatology table that the
perturbation grid is applied to.


In [ ]:
clim = (df.groupby(["SUB","MON"], as_index=False)
          .agg({
              "AREAkm2":"first",
              "PRECIPmm":"mean",
              "PETmm":"mean",
              "ETmm":"mean",
              "SWmm":"mean",
              "SURQmm":"mean",
              "WYLDmm":"mean",
              "SYLDt_ha":"mean",
          }))

clim.to_parquet(OUT_DIR / "swat_monthly_climatology.parquet", index=False)
print("✅ Saved:", OUT_DIR / "swat_monthly_climatology.parquet")
clim.head()


## 4. Fit ridge regressions per sub-basin

For each of the 71 sub-basins, three ridge regressions are fitted using the
closed-form Tikhonov solution (Equation 3.1 in the thesis):

- PRECIP, PET to SURQ (surface runoff)
- PRECIP, PET to PERC (percolation / groundwater recharge)
- log(SURQ) to log(SYLD) (sediment yield, log-log relationship)

The regularisation parameter alpha = 1e-6 is applied to the slope coefficients
but not to the intercept. Sub-basins with fewer than 24 valid annual observations
are excluded.


In [ ]:
import numpy as np

def fit_linear(X, y, ridge_alpha=1e-6):
    X = np.asarray(X, float)
    y = np.asarray(y, float).reshape(-1,1)
    X1 = np.c_[np.ones((X.shape[0], 1)), X]
    I = np.eye(X1.shape[1])
    I[0,0] = 0.0
    beta = np.linalg.inv(X1.T @ X1 + ridge_alpha*I) @ (X1.T @ y)
    return beta.flatten()

def predict_linear(beta, X):
    X = np.asarray(X, float)
    X1 = np.c_[np.ones((X.shape[0], 1)), X]
    return X1 @ beta

models = {}
eps = 1e-6

for sub, g in df.groupby("SUB"):
    g = g.dropna(subset=["PRECIPmm","PETmm","SURQmm","SWmm","SYLDt_ha"])
    if len(g) < 24:
        continue

    X = g[["PRECIPmm","PETmm"]].values
    beta_q  = fit_linear(X, g["SURQmm"].values)
    beta_sw = fit_linear(X, g["SWmm"].values)

    xsed = np.log(g["SURQmm"].values + eps)
    ysed = np.log(g["SYLDt_ha"].values + eps)
    beta_sed = fit_linear(xsed.reshape(-1,1), ysed)

    models[int(sub)] = {"beta_runoff": beta_q, "beta_sw": beta_sw, "beta_sed": beta_sed}

print("✅ Models fitted for", len(models), "subbasins:", sorted(models.keys()))


## 5. Climate perturbation grid and intervention scenarios

The perturbation grid covers three precipitation changes (0%, +10%, +20%)
and three temperature changes (0, +1, +2 C). Temperature is mapped to PET
at +4% per degree C following the Hargreaves approximation (Allen et al., 1998).

Four intervention scenarios are applied as multipliers on the climate-perturbed
values (Table 3.4 in the thesis). ET and WYLD are carried through from the
monthly climatology without regression.


In [ ]:
PET_PCT_PER_DEGC = 0.04

SCENARIO_MULTIPLIERS = {
    "baseline":  {"SURQmm":1.00, "WYLDmm":1.00, "SYLDt_ha":1.00, "SWmm":1.00},
    "marab":     {"SURQmm":0.90, "WYLDmm":0.95, "SYLDt_ha":0.75, "SWmm":1.10},
    "vallerani": {"SURQmm":0.81, "WYLDmm":0.90, "SYLDt_ha":0.50, "SWmm":1.08},
    "combined":  {"SURQmm":0.73, "WYLDmm":0.88, "SYLDt_ha":0.40, "SWmm":1.15},
}

def climate_adjust(clim_row, model, dP_pct=0.0, dT_degC=0.0):
    P0 = float(clim_row["PRECIPmm"])
    PET0 = float(clim_row["PETmm"])

    P1 = P0 * (1.0 + dP_pct)
    PET1 = PET0 * (1.0 + PET_PCT_PER_DEGC * dT_degC)

    q1  = float(predict_linear(model["beta_runoff"], [[P1, PET1]])[0])
    sw1 = float(predict_linear(model["beta_sw"],    [[P1, PET1]])[0])
    q1  = max(q1, 0.0)
    sw1 = max(sw1, 0.0)

    log_sy = float(predict_linear(model["beta_sed"], [[np.log(q1 + 1e-6)]])[0])
    sy1 = float(np.exp(log_sy) - 1e-6)
    sy1 = max(sy1, 0.0)

    return {"PRECIPmm":P1, "PETmm":PET1, "SURQmm":q1, "SWmm":sw1, "SYLDt_ha":sy1}

def apply_multipliers(vals, multipliers):
    out = vals.copy()
    for k, m in multipliers.items():
        if k in out:
            out[k] = max(out[k] * float(m), 0.0)
    return out

# what-if grid for your dashboard
dP_list = [0.0, 0.10, 0.20]   # +0%, +10%, +20% rain
dT_list = [0.0, 1.0, 2.0]     # +0C, +1C, +2C

rows = []
for _, r in clim.iterrows():
    sub = int(r["SUB"])
    mon = int(r["MON"])
    if sub not in models:
        continue

    for dP in dP_list:
        for dT in dT_list:
            base = climate_adjust(r, models[sub], dP, dT)
            base["ETmm"]   = float(r["ETmm"])
            base["WYLDmm"] = float(r["WYLDmm"])
            base["AREAkm2"] = float(r["AREAkm2"])

            for scen, mult in SCENARIO_MULTIPLIERS.items():
                vals = apply_multipliers(base, mult)
                rows.append({
                    "SUB": sub, "MON": mon,
                    "scenario": scen, "dP_pct": dP, "dT_degC": dT,
                    **vals
                })

dt_cube = pd.DataFrame(rows)

print("✅ dt_cube created")
print("Rows:", len(dt_cube))
print("SUB count:", dt_cube["SUB"].nunique(), "Scenarios:", dt_cube["scenario"].unique())
dt_cube.head()


## 6. Export the perturbation cube

Save the monthly scenario results as parquet and CSV for inspection.


In [ ]:
dt_cube.to_parquet(OUT_DIR / "dt_scenarios_monthly.parquet", index=False)
dt_cube.to_csv(OUT_DIR / "dt_scenarios_monthly.csv", index=False)

print("✅ Saved:")
print(" -", OUT_DIR / "dt_scenarios_monthly.parquet")
print(" -", OUT_DIR / "dt_scenarios_monthly.csv")

print("\nFiles inside DT_Engine:")
for p in sorted(OUT_DIR.iterdir()):
    print(" -", p.name, "|", round(p.stat().st_size/1024,2), "KB")


## 7. CMIP6 near-term proxy cases

Two CMIP6-informed proxy cases extend the perturbation grid beyond the reference values:

- SSP2-4.5 near-term: precipitation approx. -3%, temperature approx. +1.4 C
- SSP5-8.5 near-term: precipitation approx. +2%, temperature approx. +1.6 C

Source: Alsalal et al. (2024), Mujib Basin projections using bias-corrected EC-Earth3-Veg.

The method fits annual regressions on the SWAT data, predicts annual values under
each CMIP6 perturbation, computes the ratio against the reference climate, and
scales the existing monthly arrays by that ratio. This preserves the seasonal
pattern while shifting the magnitude. Intervention multipliers are then applied.


In [ ]:
# ============================================================
# CMIP6 PROXY BUILDER  -  ANNUAL REGRESSION + MONTHLY SCALING
# ============================================================
#
# Your parquets are annual-only (MON=0). So this cell:
#   1. Fits regressions on ANNUAL data (one row per subbasin per year)
#   2. Predicts annual baseline values under CMIP6 dP/dT
#   3. Computes ratio vs dP0_dT0 annual values in existing JSON
#   4. Scales existing dP0_dT0 monthly arrays by that ratio
#      (preserves seasonality, shifts magnitude)
#   5. Applies intervention multipliers
#   6. Writes ssp245_near and ssp585_near into the JSON
#
# Paste as ONE cell. No prior cells needed.
#
# Source: Alsalal et al. (2024), Mujib Basin, EC-Earth3-Veg
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from copy import deepcopy

# ============================================================
# 1. PATHS
# ============================================================
CESIUM_71 = REPO_ROOT  # Repository root containing runtime-data and swat-prepared

PARQUET_CANDIDATES = [
    REPO_ROOT / "swat-prepared" / "output_sub_FULL_FIXED2020.parquet",
    REPO_ROOT / "swat-prepared" / "output_sub_FULL_DEDUP.parquet",
    REPO_ROOT / "swat-prepared" / "output_sub_FULL.parquet",
]

PARQUET_PATH = None
for p in PARQUET_CANDIDATES:
    if p.exists():
        PARQUET_PATH = p
        break
if PARQUET_PATH is None:
    raise FileNotFoundError("No output_sub parquet found in Cesium_71/")

BASE_JSON_PATH = REPO_ROOT / "runtime-data" / "scenarios" / "scenarios_USED_BY_CESIUM_FINAL_71.json"
OUTPUT_JSON_PATH = REPO_ROOT / "runtime-data" / "scenarios" / "scenarios_USED_BY_CESIUM_FINAL_71_CMIP6_PROXY.json"

print("=== CMIP6 Proxy Builder (annual regression + monthly scaling) ===")
print(f"  Parquet: {PARQUET_PATH.name}")
print(f"  JSON:    {BASE_JSON_PATH.name}  exists={BASE_JSON_PATH.exists()}")

# ============================================================
# 2. CMIP6 DEFINITIONS
# ============================================================
CMIP6_SCENARIOS = {
    "ssp245_near": {
        "dP_pct":  -0.03,
        "dT_degC": 1.4,
        "display_label": "SSP2-4.5 near-term (2015\u20132034)",
    },
    "ssp585_near": {
        "dP_pct":  0.02,
        "dT_degC": 1.6,
        "display_label": "SSP5-8.5 near-term (2015\u20132034)",
    },
}

INTERVENTIONS = ["baseline", "marab", "vallerani", "combined"]
METRICS = ["precip", "pet", "runoff", "soil_moisture",
           "sediment", "groundwater", "vegetation"]

PET_PCT_PER_DEGC = 0.04
EPS = 1e-6

# ============================================================
# 3. LOAD ANNUAL SWAT DATA
# ============================================================
print("\n[1/6] Loading annual SWAT data...")
df = pd.read_parquet(PARQUET_PATH)
df["SUB"] = df["SUB"].astype(int)

# These are annual rows (MON=0), one per subbasin per year
required = ["SUB", "PRECIPmm", "PETmm", "SURQmm", "SWmm", "SYLDt_ha"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

for col in required[1:]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"  Rows: {len(df)}  |  Subbasins: {df['SUB'].nunique()}")

# ============================================================
# 4. FIT ANNUAL REGRESSIONS
# ============================================================
print("\n[2/6] Fitting annual regressions...")

def fit_linear(X, y, alpha=1e-6):
    X = np.asarray(X, float)
    y = np.asarray(y, float).reshape(-1, 1)
    X1 = np.c_[np.ones((X.shape[0], 1)), X]
    I = np.eye(X1.shape[1])
    I[0, 0] = 0.0
    beta = np.linalg.inv(X1.T @ X1 + alpha * I) @ (X1.T @ y)
    return beta.flatten()

def predict_linear(beta, X):
    X = np.asarray(X, float)
    X1 = np.c_[np.ones((X.shape[0], 1)), X]
    return X1 @ beta

models = {}
for sub, g in df.groupby("SUB"):
    g = g.dropna(subset=["PRECIPmm", "PETmm", "SURQmm", "SWmm", "SYLDt_ha"])
    if len(g) < 5:  # need at least 5 annual observations
        continue
    X = g[["PRECIPmm", "PETmm"]].values
    models[int(sub)] = {
        "beta_runoff": fit_linear(X, g["SURQmm"].values),
        "beta_sw":     fit_linear(X, g["SWmm"].values),
        "beta_sed":    fit_linear(
            np.log(g["SURQmm"].values + EPS).reshape(-1, 1),
            np.log(g["SYLDt_ha"].values + EPS)
        ),
        # Store mean annual values for ratio computation
        "mean_precip": float(g["PRECIPmm"].mean()),
        "mean_pet":    float(g["PETmm"].mean()),
    }

print(f"  Fitted {len(models)} subbasin models")

# ============================================================
# 5. LOAD BASE JSON AND DERIVE MULTIPLIERS
# ============================================================
print("\n[3/6] Loading base JSON...")
with open(BASE_JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

existing_keys = list(data.get("Subbasin_1", {}).get("whatifs", {}).keys())
print(f"  Subbasins: {len(data)}  |  Keys: {existing_keys}")

print("\n[4/6] Deriving intervention multipliers from dP0_dT0...")

def derive_multipliers(sub_node):
    ref = sub_node["whatifs"]["dP0_dT0"]["annual"]
    base = ref["baseline"]
    mults = {}
    for scen in ["marab", "vallerani", "combined"]:
        mults[scen] = {}
        for m in METRICS:
            bval = base.get(m, 0)
            mults[scen][m] = (ref[scen][m] / bval) if bval != 0 else 1.0
    return mults

s1_mults = derive_multipliers(data["Subbasin_1"])
for scen in ["marab", "vallerani", "combined"]:
    parts = [f"{m}={s1_mults[scen][m]:.3f}" for m in ["runoff", "sediment", "groundwater"]]
    print(f"  {scen}: {', '.join(parts)}")

# ============================================================
# 6. BUILD CMIP6 ENTRIES
# ============================================================
# Strategy:
#   For each subbasin:
#     1. Use regression to predict annual runoff, SW, sediment
#        under CMIP6 dP/dT
#     2. Compute ratio = predicted_annual / dP0_dT0_annual
#        for each regression-predicted metric
#     3. Scale the existing dP0_dT0 monthly arrays by that ratio
#        (preserves seasonality, shifts magnitude)
#     4. For precip/pet: scale by the known dP/dT directly
#     5. For groundwater/vegetation: carry from dP0_dT0 unchanged
#        (no regression model for these)
#     6. Apply intervention multipliers

print("\n[5/6] Building CMIP6 entries...")

added = 0
skipped = []

for sub_id in sorted(models.keys()):
    sub_key = f"Subbasin_{sub_id}"
    if sub_key not in data:
        skipped.append(sub_id)
        continue

    sub_node = data[sub_key]
    if "dP0_dT0" not in sub_node.get("whatifs", {}):
        skipped.append(sub_id)
        continue

    model = models[sub_id]
    mults = derive_multipliers(sub_node)
    ref_whatif = sub_node["whatifs"]["dP0_dT0"]

    for proxy_key, spec in CMIP6_SCENARIOS.items():
        dP = spec["dP_pct"]
        dT = spec["dT_degC"]

        # --- Predict annual baseline under CMIP6 climate ---
        P_mean  = model["mean_precip"]
        PET_mean = model["mean_pet"]
        P1   = P_mean * (1.0 + dP)
        PET1 = PET_mean * (1.0 + PET_PCT_PER_DEGC * dT)

        pred_runoff = max(float(predict_linear(
            model["beta_runoff"], [[P1, PET1]])[0]), 0.0)
        pred_sw = max(float(predict_linear(
            model["beta_sw"], [[P1, PET1]])[0]), 0.0)
        pred_sed = max(float(np.exp(predict_linear(
            model["beta_sed"], [[np.log(pred_runoff + EPS)]])[0]) - EPS), 0.0)

        # --- Compute scaling ratios vs dP0_dT0 baseline ---
        ref_base = ref_whatif["annual"]["baseline"]

        def safe_ratio(predicted, reference):
            if reference == 0:
                return 1.0
            return predicted / reference

        scale = {
            "precip":         (1.0 + dP),                    # direct from dP
            "pet":            (1.0 + PET_PCT_PER_DEGC * dT), # direct from dT
            "runoff":         safe_ratio(pred_runoff, ref_base["runoff"]),
            "soil_moisture":  safe_ratio(pred_sw, ref_base["soil_moisture"]),
            "sediment":       safe_ratio(pred_sed, ref_base["sediment"]),
            "groundwater":    1.0,  # no regression model
            "vegetation":     1.0,  # no regression model
        }

        # --- Scale monthly arrays from dP0_dT0 ---
        monthly_baseline = {}
        for m in METRICS:
            ref_monthly = ref_whatif["monthly"]["baseline"][m]
            monthly_baseline[m] = [max(v * scale[m], 0.0) for v in ref_monthly]

        # --- Apply intervention multipliers ---
        monthly = {"baseline": monthly_baseline}
        for scen in ["marab", "vallerani", "combined"]:
            monthly[scen] = {}
            for m in METRICS:
                ratio = mults[scen][m]
                monthly[scen][m] = [max(v * ratio, 0.0)
                                    for v in monthly_baseline[m]]

        # --- Annual sums ---
        annual = {}
        for scen in INTERVENTIONS:
            annual[scen] = {m: float(sum(monthly[scen][m])) for m in METRICS}

        # --- Write block ---
        sub_node.setdefault("whatifs", {})[proxy_key] = {
            "dP_pct":  dP,
            "dT_degC": dT,
            "annual":  annual,
            "monthly": monthly,
            "priority_by_scenario": {},
            "priority_summary_by_scenario": {},
            "meta": {
                "display_label":  spec["display_label"],
                "scenario_group": "projection_informed",
                "scenario_type":  "CMIP6-informed near-term proxy",
                "source": "Alsalal et al. (2024), EC-Earth3-Veg, Mujib Basin",
                "precip_note": "Near-term precipitation approximate (Fig. 10).",
                "reference_whatif_for_map_range": "dP0_dT0",
                "build_method": (
                    "Annual regression-direct: dP/dT passed through per-subbasin "
                    "ridge regression fitted on annual SWAT output. Monthly arrays "
                    "scaled from dP0_dT0 seasonality using predicted annual ratios."
                ),
                "dry_side_extrapolation": False,
                "annual_scale_factors": {m: round(scale[m], 4) for m in METRICS},
                "note": "Emulator scenario, not a full hydrological projection.",
            },
        }
        added += 1

print(f"  Added {added} entries ({added // len(CMIP6_SCENARIOS)} subs x {len(CMIP6_SCENARIOS)} scenarios)")
if skipped:
    print(f"  Skipped: {skipped}")

# ============================================================
# 7. VERIFY AND SAVE
# ============================================================
print("\n[6/6] Verification and save...")

s1 = data["Subbasin_1"]["whatifs"]
print("  Annual baseline values and intervention ratios:")
for wkey in ["dP0_dT0", "ssp245_near", "ssp585_near"]:
    if wkey not in s1:
        print(f"    {wkey}: NOT FOUND")
        continue
    w = s1[wkey]
    br = w["annual"]["baseline"]["runoff"]
    mr = w["annual"]["marab"]["runoff"]
    bs = w["annual"]["baseline"]["sediment"]
    ms = w["annual"]["marab"]["sediment"]
    print(f"    {wkey}:")
    print(f"      runoff:   base={br:.2f}  marab={mr:.2f}  ({mr/br*100:.1f}%)")
    print(f"      sediment: base={bs:.4f}  marab={ms:.4f}  ({ms/bs*100:.1f}%)")
    if "meta" in w and "annual_scale_factors" in w.get("meta", {}):
        sf = w["meta"]["annual_scale_factors"]
        print(f"      scale factors: runoff={sf['runoff']:.4f}  sed={sf['sediment']:.4f}  sm={sf['soil_moisture']:.4f}")

# Validate
errors = []
for sid in range(1, 72):
    skey = f"Subbasin_{sid}"
    if skey not in data:
        continue
    for pk in CMIP6_SCENARIOS:
        if pk not in data[skey].get("whatifs", {}):
            errors.append(f"{skey}: missing {pk}")
            continue
        block = data[skey]["whatifs"][pk]
        for scen in INTERVENTIONS:
            for m in METRICS:
                av = block["annual"].get(scen, {}).get(m)
                if av is None or not isinstance(av, (int, float)):
                    errors.append(f"{skey}/{pk}/annual/{scen}/{m}")
                mv = block["monthly"].get(scen, {}).get(m)
                if not isinstance(mv, list) or len(mv) != 12:
                    errors.append(f"{skey}/{pk}/monthly/{scen}/{m}")

if errors:
    print(f"\n  ERRORS ({len(errors)}):")
    for e in errors[:10]:
        print(f"    {e}")
else:
    print("\n  All entries valid.")

# Preserve default
for v in data.values():
    if isinstance(v, dict) and "default_whatif" in v:
        v["default_whatif"] = "dP0_dT0"

with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

print(f"\nSaved: {OUTPUT_JSON_PATH}")
print(f"Final keys: {list(data['Subbasin_1']['whatifs'].keys())}")
print("\nDone. Refresh your dashboard to see the CMIP6 scenarios.")

## Summary

This notebook produced the scenario JSON file used by the dashboard at runtime.
The file contains scenario outputs for 71 SWAT sub-basins across 11 climate cases
(9 reference grid + 2 CMIP6 proxy) and 4 intervention scenarios (Baseline, Marab,
Vallerani, Combined), with monthly resolution for each combination.

The ridge regression emulator reproduces the SWAT baseline with R-squared between
0.966 and 0.999 across the three indicators (see notebook 08 for validation results).

The scenario JSON is loaded by the dashboard prototype at:
`runtime-data/scenarios/scenarios_USED_BY_CESIUM_FINAL_71_CMIP6_PROXY.json`

Thesis references: Section 3.4.2 (emulator construction), Section 4.6 (scenario results),
Table 3.4 (intervention multipliers), Equations 3.1-3.6.
